
# 08 — Gold Layer: Audit Marts (Observabilidad)

Este notebook lee los datos operativos desde `tlc_audit` y construye Datamarts optimizados para el dashboard de observabilidad.
- `mart_pipeline_performance`: Estadísticas de duración y éxito por pipeline.
- `mart_data_quality_issues`: Análisis de registros descartados por motivo (cuarentena).


In [1]:

import sys
sys.path.insert(0, '/home/jovyan/work')

from src.spark_utils import get_spark, read_mongo, write_mongo
from core.config.settings import settings
import pyspark.sql.functions as F

print('Imports OK.')


Imports OK.


In [2]:

spark = get_spark('TLC_Gold_Audit_Marts')
spark.conf.set('spark.sql.shuffle.partitions', '20')  # Audit data is much smaller, 20 is enough
print(f'Spark {spark.version} ready.')


2026-07-20 00:12:11 | INFO     | tlc.spark_utils | [SPARK] Session ready | version=3.4.1 master=local[*]
Spark 3.4.1 ready.



---
## Mart 1: `mart_pipeline_performance`
Agrupa los `pipeline_runs` por fecha de ejecución y pipeline.


In [3]:

print('Procesando mart_pipeline_performance...')

# Leer pipeline_runs
df_runs = read_mongo(spark, settings.MONGO_DB_AUDIT, 'pipeline_runs')

# Extraer métricas y agregar
mart_performance = (
    df_runs
    .withColumn("run_date", F.to_date("start_time"))
    .groupBy("run_date", "pipeline_name", "status")
    .agg(
        F.count("*").alias("total_runs"),
        F.avg("duration_seconds").alias("avg_duration_seconds"),
        F.max("duration_seconds").alias("max_duration_seconds"),
        F.min("duration_seconds").alias("min_duration_seconds"),
        F.sum(F.coalesce(F.col("output_summary.records_written_to_silver"), F.col("output_summary.total_records_written"), F.lit(0)).cast("long")).alias("total_rows_written"),
        F.sum(F.coalesce(F.col("output_summary.records_quarantined"), F.lit(0)).cast("long")).alias("total_rows_rejected"),
    )
    .orderBy(F.col("run_date").desc(), F.col("pipeline_name"))
)

write_mongo(mart_performance, settings.MONGO_DB_GOLD, 'mart_pipeline_performance', mode='overwrite')
print('✓ mart_pipeline_performance escrito exitosamente')


Procesando mart_pipeline_performance...
2026-07-20 00:12:12 | INFO     | tlc.spark_utils | [SPARK] Read from MongoDB tlc_audit.pipeline_runs
2026-07-20 00:12:14 | INFO     | tlc.spark_utils | [SPARK] Wrote Unknown rows → MongoDB tlc_gold.mart_pipeline_performance (mode=overwrite)
✓ mart_pipeline_performance escrito exitosamente



---
## Mart 2: `mart_data_quality_issues`
Agrupa los registros de `quarantine` por pipeline y motivo de rechazo.


In [4]:

print('Procesando mart_data_quality_issues...')

# Leer quarantine
df_quarantine = read_mongo(spark, settings.MONGO_DB_AUDIT, 'quarantine')

# Validar si quarantine está vacía
if df_quarantine.count() > 0:
    mart_quality = (
        df_quarantine
        .withColumn("quarantine_date", F.to_date("quarantined_at"))
        .groupBy("quarantine_date", "pipeline", "reason")
        .agg(
            F.count("*").alias("rejected_records")
        )
        .orderBy(F.col("quarantine_date").desc(), F.col("rejected_records").desc())
    )
    
    write_mongo(mart_quality, settings.MONGO_DB_GOLD, 'mart_data_quality_issues', mode='overwrite')
    print('✓ mart_data_quality_issues escrito exitosamente')
else:
    print('⚠ La colección quarantine está vacía. No se escribió el datamart.')


Procesando mart_data_quality_issues...
2026-07-20 00:12:15 | INFO     | tlc.spark_utils | [SPARK] Read from MongoDB tlc_audit.quarantine
2026-07-20 00:13:25 | INFO     | tlc.spark_utils | [SPARK] Wrote Unknown rows → MongoDB tlc_gold.mart_data_quality_issues (mode=overwrite)
✓ mart_data_quality_issues escrito exitosamente
